<a href="https://colab.research.google.com/github/HeathSkinner458/DET-520/blob/main/DET_520_final_518_no_kaggle_connection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
#need to lock in our state
import os, random
import numpy as np
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
import tensorflow as tf
import keras
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [83]:
#ok, so i realized that to pull the data you needed kaggle creds, so I am pulling the disaster tweet data from a public repository in this version
import pandas as pd
BASE = "https://raw.githubusercontent.com/sugatagh/Natural-Language-Processing-with-Disaster-Tweets/main/train.csv"
train = pd.read_csv(BASE)
print("train shape", train.shape)

train shape (7613, 5)


In [84]:
#lets see what the data looks like
df = train.copy()
print(df.shape)
df.head()

(7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [86]:
#ok, pre-processing. Firstly based on the DF missing quite a bit of the location data. Does the presence of location affect ability to predict target?
df['has_location'] = df['location'].notna().astype(int)
print(df.groupby('has_location')['target'].mean())
print(df.groupby('has_location').size())

has_location
0    0.424398
1    0.432283
Name: target, dtype: float64
has_location
0    2533
1    5080
dtype: int64


In [87]:
#Location yes/no is negative, will drop for parsimony. Lets move on to the keywords. How many and how much?
df['keyword'].nunique()

221

In [88]:
df['keyword'].value_counts().head(20)

,count
keyword,
fatalities,45
deluge,42
armageddon,42
damage,41
body%20bags,41
harm,41
sinking,41
evacuate,40
outbreak,40


In [89]:
#are the individual keywords associated with target
keyword_stats = df.groupby('keyword')['target'].agg(['mean','std', 'count']).sort_values('mean', ascending=False)
print(keyword_stats)

                 mean       std  count
keyword                               
derailment   1.000000  0.000000     39
debris       1.000000  0.000000     37
wreckage     1.000000  0.000000     39
outbreak     0.975000  0.158114     40
typhoon      0.973684  0.162221     38
...               ...       ...    ...
body%20bag   0.030303  0.174078     33
blazing      0.029412  0.171499     34
ruin         0.027027  0.164399     37
body%20bags  0.024390  0.156174     41
aftershock   0.000000  0.000000     34

[221 rows x 3 columns]


In [90]:
#so there seems to be an associated between target and keyword, however if the keyword is in the text, we may not need to make this more
#complicated with another variable. So lets look at the overlap.
def keyword_in_text(row):
  if pd.isna(row['keyword']):
    return None
  kw = row['keyword'].replace('%20', ' ').lower()
  return kw in row['text'].lower()

df['keyword_in_text'] = df.apply(keyword_in_text, axis=1)
print("Fraction of rows where keyword appears in text: ", df['keyword_in_text'].mean())
print()
print("breakdown:")
print(df['keyword_in_text'].value_counts(dropna=False))

Fraction of rows where keyword appears in text:  0.8871822033898306

breakdown:
keyword_in_text
True     6700
False     852
None       61
Name: count, dtype: int64


In [91]:
#Big overlap, so is the absence of exact keyword in the text associated with target (combine none and false vs true)?
df['kw_signal_visible'] = df['keyword_in_text'] == True
print("Disaster rate by keyword visible:")
print(df.groupby('kw_signal_visible')['target'].mean())
print ("counts:")
print(df.groupby('kw_signal_visible').size())

Disaster rate by keyword visible:
kw_signal_visible
False    0.431544
True     0.429403
Name: target, dtype: float64
counts:
kw_signal_visible
False     913
True     6700
dtype: int64


In [92]:
#ok, high overlap and no real info in keyword so will omit from model.
#now to making our train and validation sets (differnet from the 'test" set in kaggle, which took a bit to realize)
import sklearn

In [93]:
#so 80/20 split. We locked the random state above (after some element of debugging) but doesnt hurt anything to leave at this point.
#also need to stratify by target
train_df, val_df = sklearn.model_selection.train_test_split(df,test_size=0.2, random_state=42, stratify=df['target'])

In [94]:
#did it work? numbers
print(train_df.shape)
print(val_df.shape)

(6090, 8)
(1523, 8)


In [95]:
#did it work? target equal between train and val
print (train_df['target'].mean())
print (val_df['target'].mean())

0.4297208538587849
0.4294156270518713


In [96]:
#ok, we need to figure out how to handle special aspects of the tweets themselves. firstly, URLs
#lets see how predictive and how many
def has_url(row):
  if pd.isna(row['text']):
    return None
  return 'http' in row['text']

train_df['has_url'] = train_df.apply(has_url, axis=1)
print(train_df.groupby('has_url')['target'].mean())
print(train_df.groupby('has_url').size())

has_url
False    0.299897
True     0.548600
Name: target, dtype: float64
has_url
False    2911
True     3179
dtype: int64


In [97]:
#ok, value in URL yes no
#lets look at hashtags the same way
def has_hashtag(row):
  if pd.isna(row['text']):
    return None
  return '#' in row['text']

train_df['has_hashtag'] = train_df.apply(has_hashtag, axis=1)
print(train_df.groupby('has_hashtag')['target'].mean())
print(train_df.groupby('has_hashtag').size())

has_hashtag
False    0.407906
True     0.502128
Name: target, dtype: float64
has_hashtag
False    4680
True     1410
dtype: int64


In [98]:
#ok, some value in hashtags and enough to matter I think
#now lets look at @ (mentions)
def has_mention(row):
  if pd.isna(row['text']):
    return None
  return '@' in row['text']

train_df['has_mention'] = train_df.apply(has_mention, axis=1)
print(train_df.groupby('has_mention')['target'].mean())
print(train_df.groupby('has_mention').size())

has_mention
False    0.463295
True     0.337238
Name: target, dtype: float64
has_mention
False    4468
True     1622
dtype: int64


In [99]:
#value there, same as in others and enough to be meaningful I think
#same for numbers
def has_number(row):
  if pd.isna(row['text']):
    return None
  return any(char.isdigit() for char in row['text'])

train_df['has_number'] = train_df.apply(has_number, axis=1)
print(train_df.groupby('has_number')['target'].mean())
print(train_df.groupby('has_number').size())

has_number
False    0.289505
True     0.526506
Name: target, dtype: float64
has_number
False    2487
True     3603
dtype: int64


In [100]:
#value there
#ok, caps here is a bit different. My hypothesis here is that generally it wont matter, but will for things like FIRE or FLOOD
#so, lets look at at least two characters in a row caps
def has_allcaps(row):
  if pd.isna(row['text']):
    return None
  return any(len(word)>=2 and word.isupper() for word in row['text'].split())

train_df['has_allcaps'] = train_df.apply(has_allcaps, axis=1)
print(train_df.groupby('has_allcaps')['target'].mean())
print(train_df.groupby('has_allcaps').size())

has_allcaps
False    0.402775
True     0.492107
Name: target, dtype: float64
has_allcaps
False    4253
True     1837
dtype: int64


In [26]:
#value there
#now emojis same way
!pip install emoji
import emoji
import pandas as pd

def has_emoji(row):
  if pd.isna(row['text']):
    return None
  return any(emoji.is_emoji(char) for char in row['text'])

train_df['has_emoji'] = train_df.apply(has_emoji, axis=1)
print(train_df.groupby('has_emoji')['target'].mean())
print(train_df.groupby('has_emoji').size())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.6 MB/s eta 0:00:00
has_emoji
False    0.429041
True     0.888889
Name: target, dtype: float64
has_emoji
False    6081
True        9
dtype: int64


In [27]:
#EMojis "predicive" but so few as to be meaningless. IN service of parsimony we can drop i think
#same for exclamation
def has_exclamation(row):
  if pd.isna(row['text']):
    return None
  return '!' in row['text']

train_df['has_exclamation'] = train_df.apply(has_exclamation, axis=1)
print(train_df.groupby('has_exclamation')['target'].mean())
print(train_df.groupby('has_exclamation').size())

has_exclamation
False    0.445877
True     0.277397
Name: target, dtype: float64
has_exclamation
False    5506
True      584
dtype: int64


In [28]:
#just to look at them altogether
features = ['has_location', 'has_url', 'has_hashtag','has_exclamation','has_mention', 'has_number', 'has_allcaps', 'has_emoji']
for feat in features:
  grouped = train_df.groupby(feat)['target'].agg(['mean', 'count'])
  print(grouped)
  print()

                  mean  count
has_location                 
0             0.428287   2008
1             0.430426   4082

             mean  count
has_url                 
False    0.299897   2911
True     0.548600   3179

                 mean  count
has_hashtag                 
False        0.407906   4680
True         0.502128   1410

                     mean  count
has_exclamation                 
False            0.445877   5506
True             0.277397    584

                 mean  count
has_mention                 
False        0.463295   4468
True         0.337238   1622

                mean  count
has_number                 
False       0.289505   2487
True        0.526506   3603

                 mean  count
has_allcaps                 
False        0.402775   4253
True         0.492107   1837

               mean  count
has_emoji                 
False      0.429041   6081
True       0.888889      9



In [29]:
#we are not using emojis because they are so few
#Ok now we need to clean the training data here is my process:
#1. Replace ALL caps word with <allcaps>
#2. lowercase all words
#3. Replace URLs with <httpurl>
#4. Replace @mentions with <usermentions>
#5. Replace #hashtags with <hashtag>
#6. Replace numbers with <numtoken>
#7. Replace ! with <exclaim>
#8. Strip remaining puncuation
#9. collpase multiple spaces into one
import re

In [30]:
#lets compile what each catagory means
ALLCAPS_RE = re.compile(r"\b[A-Z]{2,}\b")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
NUMBER_RE = re.compile(r"\b\d+\.?\d*\b")
EXCLAIM_RE = re.compile(r"!")
MULTI_SPACE = re.compile(r"\s+")
#


In [31]:
#now we define the clean text
def clean_text(s):
  s = ALLCAPS_RE.sub(" allcaps ", s)
  s = s.lower()
  s = URL_RE.sub(" httpurl ", s)
  s = MENTION_RE.sub(" usermentions ", s)
  s = HASHTAG_RE.sub(r" hashtag ", s)
  s = NUMBER_RE.sub(" numtoken ", s)
  s = EXCLAIM_RE.sub(" exclaim ", s)
  s = MULTI_SPACE.sub(" ", s).strip()
  return s

In [32]:
# testing the clean text function
samples = ["https://pubmed.ncbi.nlm.nih.gov/"
"@HEATH SKINNER"
"I am tired of this!!!!!"]
for s in samples:
  print(clean_text(s))

httpurl allcaps allcaps am tired of this exclaim exclaim exclaim exclaim exclaim


In [33]:
#ok seems to work. lets clean both sets of data
train_df['clean'] = train_df['text'].apply(clean_text)
val_df['clean'] = val_df['text'].apply(clean_text)

In [34]:
#lets check for anything weird
train_df[['text','clean']].head(10)

,text,clean
6234,Sassy city girl country hunk stranded in Smoky...,sassy city girl country hunk stranded in smoky...
326,God's Kingdom (Heavenly Gov't) will rule over ...,god's kingdom (heavenly gov't) will rule over ...
997,Mopheme and Bigstar Johnson are a problem in t...,mopheme and bigstar johnson are a problem in t...
7269,@VixMeldrew sounds like a whirlwind life!,usermentions sounds like a whirlwind life exclaim
2189,Malaysia confirms plane debris washed up on Re...,malaysia confirms plane debris washed up on re...
3725,Live a balanced life! Balance your fear of #Al...,live a balanced life exclaim balance your fear...
4062,CLIMATE CONSEQUENCES: U.S. Forest Service Says...,allcaps allcaps : u.s. forest service says spe...
5928,brooke just face timed me at the concert and j...,brooke just face timed me at the concert and j...
5178,'But time began at last to obliterate the fres...,'but time began at last to obliterate the fres...
5079,What Natural Disaster Are You When You Get Ang...,what natural disaster are you when you get ang...


In [35]:
# Lets convert the text to vectors and check ourselves
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3), min_df=2)
x_train_tfidf = vectorizer.fit_transform(train_df['clean'])
X_val_tfidf = vectorizer.transform(val_df['clean'])

In [36]:
#everything make sense?
print("Training matrix shape", x_train_tfidf.shape)
print("Validation matrix shape", X_val_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Training matrix shape (6090, 10000)
Validation matrix shape (1523, 10000)
Vocabulary size: 10000


In [37]:
#now run logistic regression for a baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(x_train_tfidf, train_df['target'])

LogisticRegression(max_iter=1000, random_state=42)

In [38]:
#now show me the LR prediction baseline
val_predicted = lr.predict(X_val_tfidf)
Accuracy = accuracy_score(val_df['target'], val_predicted)
Precision = precision_score(val_df['target'], val_predicted)
Recall = recall_score(val_df['target'], val_predicted)
F1 = f1_score(val_df['target'], val_predicted)

In [39]:
#print a table of the LR model
import pandas as pd
results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [Accuracy],
    "Precision": [Precision],
    "Recall": [Recall],
    "F1": [F1]
})
print(results)

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.801051   0.811723  0.698777  0.751027


In [40]:
#Building our token IDS
from collections import Counter
import numpy as np
PAD_ID = 0
UNK_ID = 1
MAX_LEN = 40
MIN_FREQ = 2
VOCAB_SIZE_CAP = 10000
#count training word freq
counter = Counter()
for tweet in train_df['clean']:
  counter.update(tweet.split())

In [41]:
#cap tokens, retain those above min reserve 2 slots for pad and unk
common = [[tok,c] for tok, c in counter.most_common() if c>=MIN_FREQ]
common = common[:VOCAB_SIZE_CAP - 2]

In [42]:
#build vocab
vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for tok, _ in common:
  vocab[tok] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"Vocab size:, {VOCAB_SIZE}")

Vocab size:, 5446


In [43]:
#now the encode
def encode(text, max_len=MAX_LEN):
  ids = [vocab.get(tok, UNK_ID) for tok in text.split()]
  ids = ids[:max_len]
  ids = [PAD_ID] * (max_len - len(ids)) + ids
  return ids

In [44]:
#lets test it
sample = train_df['clean'].iloc[0]
print("Cleaned text:", sample)
print("Encoded:", encode(sample))
print("Length:", len(encode(sample)))

Cleaned text: sassy city girl country hunk stranded in smoky mountain snowstorm hashtag httpurl hashtag hashtag
Encoded: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3529, 193, 366, 770, 3530, 2131, 10, 3531, 1009, 835, 5, 3, 5, 5]
Length: 40


In [45]:
#ARRAYS!
x_train = np.array([encode(t) for t in train_df['clean']])
x_val = np.array([encode(t) for t in val_df['clean']])
y_train = train_df['target'].values
y_val = val_df['target'].values

In [46]:
#have we lost anything?
print(x_train.shape)
print(x_val.shape)
print(y_train.shape)
print(y_val.shape)

(6090, 40)
(1523, 40)
(6090,)
(1523,)


In [47]:
#ok, seems like we need to go keras now?
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras import layers

In [48]:
#here is our first model FFNN
model = keras.Sequential([keras.Input(shape=(MAX_LEN,)), layers.Embedding(VOCAB_SIZE, 64),
                          layers.GlobalAveragePooling1D(), layers.Dense(64,activation='relu'),
                          layers.Dense(1, activation='sigmoid')
                          ])

In [49]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │       348,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 352,769 (1.35 MB)

 Trainable params: 352,769 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

In [50]:
#now we need to optimize the model
model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy'])

In [51]:
#ok we need to make sure we dont overfit (hence the early stopping call back) and save the non-overfitted model (restore best weights)
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(x_train, y_train, batch_size=32, epochs=10,
                    callbacks=[early_stopping], validation_data=(x_val, y_val), verbose=2)

Epoch 1/10
191/191 - 8s - 40ms/step - accuracy: 0.6199 - loss: 0.6474 - val_accuracy: 0.6645 - val_loss: 0.6016
Epoch 2/10
191/191 - 3s - 14ms/step - accuracy: 0.7650 - loss: 0.5044 - val_accuracy: 0.7498 - val_loss: 0.5047
Epoch 3/10
191/191 - 6s - 32ms/step - accuracy: 0.8199 - loss: 0.4069 - val_accuracy: 0.7728 - val_loss: 0.4784
Epoch 4/10
191/191 - 2s - 8ms/step - accuracy: 0.8502 - loss: 0.3487 - val_accuracy: 0.7800 - val_loss: 0.4783
Epoch 5/10
191/191 - 1s - 6ms/step - accuracy: 0.8700 - loss: 0.3078 - val_accuracy: 0.7905 - val_loss: 0.4931
Epoch 6/10
191/191 - 1s - 6ms/step - accuracy: 0.8844 - loss: 0.2769 - val_accuracy: 0.7820 - val_loss: 0.5165
Epoch 7/10
191/191 - 1s - 7ms/step - accuracy: 0.8957 - loss: 0.2536 - val_accuracy: 0.7794 - val_loss: 0.5427


In [52]:
#ok, lets set the threshold
FFNN_val_pred_prob = model.predict(x_val).flatten()
FFNN_val_preds = (FFNN_val_pred_prob >= 0.5).astype(int)

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [53]:
#print the metrics
print("FFNN Validation Accuracy:", accuracy_score(y_val, FFNN_val_preds))
print("FFNN Validation Precision:", precision_score(y_val, FFNN_val_preds))
print("FFNN Validation Recall:", recall_score(y_val, FFNN_val_preds))
print("FFNN Validation F1 Score:", f1_score(y_val, FFNN_val_preds))
#

FFNN Validation Accuracy: 0.7800393959290873
FFNN Validation Precision: 0.7314949201741655
FFNN Validation Recall: 0.7706422018348624
FFNN Validation F1 Score: 0.7505584512285927


In [54]:
#add the FFNN model to the data frame for visualization
new_row = pd.DataFrame({
    "Model": ["FFNN"],
    "Accuracy": [accuracy_score(y_val, FFNN_val_preds)],
    "Precision": [precision_score(y_val, FFNN_val_preds)],
    "Recall": [recall_score(y_val, FFNN_val_preds)],
    "F1": [f1_score(y_val, FFNN_val_preds)]
})
results = pd.concat([results, new_row], ignore_index=True)
print(results)

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.801051   0.811723  0.698777  0.751027
1                 FFNN  0.780039   0.731495  0.770642  0.750558


In [55]:
#Ok lets do our CNN example, we are going to use the same embedding as the FFNN, but add sequence using bigrams and trigrams and penta
from tensorflow import keras
from tensorflow.keras import layers

In [56]:
#set our parameters
VOCAB_SIZE = 5446
MAX_LEN = 40
EMBED_DIM = 64
N_FILTERS = 64

In [57]:
#set up the model and visualize the summary
inputs = keras.Input(shape=(MAX_LEN,))
x = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(inputs)
branch1 = layers.Conv1D(N_FILTERS, kernel_size=2, activation='relu')(x)
branch1 = layers.GlobalMaxPooling1D()(branch1)
branch2 = layers.Conv1D(N_FILTERS, kernel_size=3, activation='relu')(x)
branch2 = layers.GlobalMaxPooling1D()(branch2)
branch3 = layers.Conv1D(N_FILTERS, kernel_size=5, activation='relu')(x)
branch3 = layers.GlobalMaxPooling1D()(branch3)
combined =  layers.concatenate([branch1, branch2, branch3])
combined = layers.Dropout(0.5)(combined)
outputs = layers.Dense(1, activation='sigmoid')(combined)
cnn_model = keras.Model(inputs=inputs, outputs=outputs, name = "cnn_parallel_model")
cnn_model.summary()

Model: "cnn_parallel_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 40, 64)    │    348,544 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 39, 64)    │      8,256 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 38, 64)    │     12,352 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 36, 64)    │     20,544 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 192)       │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        193 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 389,889 (1.49 MB)

 Trainable params: 389,889 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [58]:
#same compilation as with FFNN
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [59]:
#same eval as FFNN
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
cnn_history = cnn_model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=32, epochs=10, callbacks=early_stopping, verbose=2)


Epoch 1/10
191/191 - 5s - 26ms/step - accuracy: 0.6754 - loss: 0.6012 - val_accuracy: 0.7807 - val_loss: 0.4873
Epoch 2/10
191/191 - 4s - 21ms/step - accuracy: 0.8144 - loss: 0.4141 - val_accuracy: 0.7886 - val_loss: 0.4693
Epoch 3/10
191/191 - 4s - 21ms/step - accuracy: 0.8700 - loss: 0.3119 - val_accuracy: 0.7748 - val_loss: 0.5223
Epoch 4/10
191/191 - 3s - 15ms/step - accuracy: 0.9076 - loss: 0.2353 - val_accuracy: 0.7577 - val_loss: 0.6060
Epoch 5/10
191/191 - 4s - 19ms/step - accuracy: 0.9322 - loss: 0.1826 - val_accuracy: 0.7597 - val_loss: 0.6914


In [60]:
#lets get our CNN metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
CNN_val_pred_prob = cnn_model.predict(x_val).flatten()
CNN_val_preds = (CNN_val_pred_prob >= 0.5).astype(int)

48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


In [61]:
#and print them
CNN_metrics = {
    "Model": ["CNN"],
    "Accuracy": [accuracy_score(y_val, CNN_val_preds)],
    "Precision": [precision_score(y_val, CNN_val_preds)],
    "Recall": [recall_score(y_val, CNN_val_preds)],
    "F1": [f1_score(y_val, CNN_val_preds)]
}
print(CNN_metrics)

{'Model': ['CNN'], 'Accuracy': [0.788575180564675], 'Precision': [0.7507552870090635], 'Recall': [0.7599388379204893], 'F1': [0.7553191489361702]}


In [62]:
#an dput them in the dataframe
import pandas as pd
new_row = pd.DataFrame(CNN_metrics)
results = pd.concat([results, new_row], ignore_index=True)
print(results)

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.801051   0.811723  0.698777  0.751027
1                 FFNN  0.780039   0.731495  0.770642  0.750558
2                  CNN  0.788575   0.750755  0.759939  0.755319


In [63]:
#final individual model Deo gratis
inputs = keras.Input(shape=(MAX_LEN,), dtype="int32")
#embedd like other models. masking here is important to denote
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, mask_zero=True)(inputs)
#this goes forward and backward 64 units
x = layers.Bidirectional(layers.LSTM(64))(x)
#drop out helps decrease overfitting
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

In [64]:
#Avengers assemble!
bi_lstm_model = keras.Model(inputs=inputs, outputs=outputs, name="bi_lstm_model")
bi_lstm_model.summary()

Model: "bi_lstm_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 40, 64)    │    348,544 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 40)        │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     66,048 │ embedding_2[0][0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        129 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 414,721 (1.58 MB)

 Trainable params: 414,721 (1.58 MB)

 Non-trainable params: 0 (0.00 B)

In [65]:
#compile and add early stopping
bi_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
bi_lstm_model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=32, epochs=10, callbacks=[early_stopping], verbose=2)

Epoch 1/10
191/191 - 16s - 84ms/step - accuracy: 0.7228 - loss: 0.5516 - val_accuracy: 0.7965 - val_loss: 0.4619
Epoch 2/10
191/191 - 11s - 58ms/step - accuracy: 0.8356 - loss: 0.3750 - val_accuracy: 0.8004 - val_loss: 0.4598
Epoch 3/10
191/191 - 10s - 53ms/step - accuracy: 0.8818 - loss: 0.2900 - val_accuracy: 0.7603 - val_loss: 0.5390
Epoch 4/10
191/191 - 10s - 53ms/step - accuracy: 0.9117 - loss: 0.2305 - val_accuracy: 0.7426 - val_loss: 0.6162
Epoch 5/10
191/191 - 11s - 60ms/step - accuracy: 0.9299 - loss: 0.1889 - val_accuracy: 0.7328 - val_loss: 0.6374


In [66]:
# lets get the metrics and print them
bi_lstm_model_y_val_pred_prob = bi_lstm_model.predict(x_val).flatten()
bi_lstm_model_y_val = (bi_lstm_model_y_val_pred_prob >= 0.5).astype(int)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
bi_lstm_model_metrics = {
    "Model": ["Bi-LSTM"],
    "Accuracy": [accuracy_score(y_val, bi_lstm_model_y_val)],
    "Precision": [precision_score(y_val, bi_lstm_model_y_val)],
    "Recall":[recall_score(y_val,bi_lstm_model_y_val)],
    "F1":[f1_score(y_val,bi_lstm_model_y_val)]}
print(bi_lstm_model_metrics)


48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
{'Model': ['Bi-LSTM'], 'Accuracy': [0.8003939592908733], 'Precision': [0.7859477124183006], 'Recall': [0.735474006116208], 'F1': [0.7598736176935229]}


In [67]:
#lets add bilstm to our dataframe
import pandas as pd
new_row = pd.DataFrame(bi_lstm_model_metrics)
results = pd.concat([results, new_row], ignore_index=True)
print(results)

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.801051   0.811723  0.698777  0.751027
1                 FFNN  0.780039   0.731495  0.770642  0.750558
2                  CNN  0.788575   0.750755  0.759939  0.755319
3              Bi-LSTM  0.800394   0.785948  0.735474  0.759874


In [68]:
# lets generate an ensemble model, first lets make sure we have what we need saved
print("FFN:", FFNN_val_pred_prob.shape, FFNN_val_pred_prob[:3])
print("CNN:", CNN_val_pred_prob.shape, CNN_val_pred_prob[:3])
print("Bi-LSTM:", bi_lstm_model_y_val_pred_prob.shape, bi_lstm_model_y_val_pred_prob[:3])


FFN: (1523,) [0.6084492  0.14133298 0.92554814]
CNN: (1523,) [0.6470349  0.32839885 0.99316365]
Bi-LSTM: (1523,) [0.4941125  0.19572555 0.94422644]


In [69]:
# now average the models
ensemble_val_pred_prob = (FFNN_val_pred_prob + CNN_val_pred_prob + bi_lstm_model_y_val_pred_prob) / 3
ensemble_val_preds = (ensemble_val_pred_prob >= 0.5).astype(int)

In [70]:
#get and print the metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
ensemble_metrics = {
    "Model": ["Ensemble"],
    "Accuracy": [accuracy_score(y_val, ensemble_val_preds)],
    "Precision": [precision_score(y_val, ensemble_val_preds)],
    "Recall": [recall_score(y_val, ensemble_val_preds)],
    "F1": [f1_score(y_val, ensemble_val_preds)]
}
print(ensemble_metrics)

{'Model': ['Ensemble'], 'Accuracy': [0.7898883782009193], 'Precision': [0.7585139318885449], 'Recall': [0.7492354740061162], 'F1': [0.7538461538461538]}


In [71]:
#lets add our average ensemble to our dataframe
import pandas as pd
new_row = pd.DataFrame(ensemble_metrics)
results = pd.concat([results, new_row], ignore_index=True)
print(results)

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.801051   0.811723  0.698777  0.751027
1                 FFNN  0.780039   0.731495  0.770642  0.750558
2                  CNN  0.788575   0.750755  0.759939  0.755319
3              Bi-LSTM  0.800394   0.785948  0.735474  0.759874
4             Ensemble  0.789888   0.758514  0.749235  0.753846


In [72]:
#well that was not helpful. Lets try a different way to ensemble: Majority
preds_stack = np.stack([FFNN_val_preds.astype(int), CNN_val_preds.astype(int), bi_lstm_model_y_val.astype(int)])
maj_ensemble_val_preds = (preds_stack.sum(axis=0) >= 2).astype(int)
#

In [73]:
#majority ensemble metrics and print
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
majority_ensemble_metrics = {
    "Model": ["Majority Ensemble"],
    "Accuracy": [accuracy_score(y_val, maj_ensemble_val_preds)],
    "Precision": [precision_score(y_val, maj_ensemble_val_preds)],
    "Recall": [recall_score(y_val, maj_ensemble_val_preds)],
    "F1": [f1_score(y_val, maj_ensemble_val_preds)]
}
print(majority_ensemble_metrics)

{'Model': ['Majority Ensemble'], 'Accuracy': [0.7918581746552856], 'Precision': [0.7588325652841782], 'Recall': [0.7553516819571865], 'F1': [0.757088122605364]}


In [74]:
#add majority metrics to dataframe
import pandas as pd
new_row = pd.DataFrame(majority_ensemble_metrics)
results = pd.concat([results, new_row], ignore_index=True)
print(results)

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.801051   0.811723  0.698777  0.751027
1                 FFNN  0.780039   0.731495  0.770642  0.750558
2                  CNN  0.788575   0.750755  0.759939  0.755319
3              Bi-LSTM  0.800394   0.785948  0.735474  0.759874
4             Ensemble  0.789888   0.758514  0.749235  0.753846
5    Majority Ensemble  0.791858   0.758833  0.755352  0.757088


In [75]:
# maybe the geometric mean?
GM_ensemble_val_pred_prob = (FFNN_val_pred_prob * CNN_val_pred_prob * bi_lstm_model_y_val_pred_prob) ** (1/3)
GM_ensemble_val_preds = (GM_ensemble_val_pred_prob >= 0.5).astype(int)

In [76]:
#GM ensemble metrics and print
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
GM_ensemble_metrics = {
    "Model": ["Geometric Mean Ensemble"],
    "Accuracy": [accuracy_score(y_val, GM_ensemble_val_preds)],
    "Precision": [precision_score(y_val, GM_ensemble_val_preds)],
    "Recall": [recall_score(y_val, GM_ensemble_val_preds)],
    "F1": [f1_score(y_val, GM_ensemble_val_preds)]
}
print(GM_ensemble_metrics)

{'Model': ['Geometric Mean Ensemble'], 'Accuracy': [0.7918581746552856], 'Precision': [0.7645211930926217], 'Recall': [0.7446483180428135], 'F1': [0.7544539116963594]}


In [77]:
import pandas as pd
new_row = pd.DataFrame(GM_ensemble_metrics)
results = pd.concat([results, new_row], ignore_index=True)
print(results)

                     Model  Accuracy  Precision    Recall        F1
0      Logistic Regression  0.801051   0.811723  0.698777  0.751027
1                     FFNN  0.780039   0.731495  0.770642  0.750558
2                      CNN  0.788575   0.750755  0.759939  0.755319
3                  Bi-LSTM  0.800394   0.785948  0.735474  0.759874
4                 Ensemble  0.789888   0.758514  0.749235  0.753846
5        Majority Ensemble  0.791858   0.758833  0.755352  0.757088
6  Geometric Mean Ensemble  0.791858   0.764521  0.744648  0.754454


In [78]:
# final step, lets try to change the threshold
from sklearn.metrics import precision_recall_curve
prec, rec, thresh = precision_recall_curve(y_val, ensemble_val_pred_prob)
f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1_curve.argmax()
print("Best threshold:", thresh[best_idx])
print("Best F1:", f1_curve[best_idx])

Best threshold: 0.48773816
Best F1: 0.7570449347627644


In [79]:
# no real difference. not worth it to change I think.

In [80]:
# final step, lets try to change the threshold for GM?
from sklearn.metrics import precision_recall_curve
prec, rec, thresh = precision_recall_curve(y_val, GM_ensemble_val_pred_prob)
f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1_curve.argmax()
print("Best threshold:", thresh[best_idx])
print("Best F1:", f1_curve[best_idx])

Best threshold: 0.48901764
Best F1: 0.7572964664738969


In [81]:
# again no improvement. I am done.